In [1]:
import pandas as pd
import numpy as np

print("Kernel funcionando")
print(pd.__version__)

Kernel funcionando
2.3.3


In [3]:
from pyspark.sql import SparkSession

In [5]:

spark = SparkSession.builder \
    .appName("AMEX_Default_Prediction") \
    .master("yarn") \
    .getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/14 07:19:36 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/05/14 07:19:43 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/05/14 07:19:58 WARN Client: Neither spark.yarn.jars nor spark.yarn.archive is set, falling back to uploading libraries under SPARK_HOME.


In [7]:
df = spark.read.csv(
    "hdfs://nn1:9000/amex/raw/train_data.csv",
    header=True,
    inferSchema=True
)

labels = spark.read.csv(
    "hdfs://nn1:9000/amex/raw/train_labels.csv",
    header=True,
    inferSchema=True
)

In [8]:
print(df.count())

5531451


In [9]:
len(df.columns)

190

In [10]:
df.printSchema()

root
 |-- customer_ID: string (nullable = true)
 |-- S_2: date (nullable = true)
 |-- P_2: double (nullable = true)
 |-- D_39: double (nullable = true)
 |-- B_1: double (nullable = true)
 |-- B_2: double (nullable = true)
 |-- R_1: double (nullable = true)
 |-- S_3: double (nullable = true)
 |-- D_41: double (nullable = true)
 |-- B_3: double (nullable = true)
 |-- D_42: double (nullable = true)
 |-- D_43: double (nullable = true)
 |-- D_44: double (nullable = true)
 |-- B_4: double (nullable = true)
 |-- D_45: double (nullable = true)
 |-- B_5: double (nullable = true)
 |-- R_2: double (nullable = true)
 |-- D_46: double (nullable = true)
 |-- D_47: double (nullable = true)
 |-- D_48: double (nullable = true)
 |-- D_49: double (nullable = true)
 |-- B_6: double (nullable = true)
 |-- B_7: double (nullable = true)
 |-- B_8: double (nullable = true)
 |-- D_50: double (nullable = true)
 |-- D_51: double (nullable = true)
 |-- B_9: double (nullable = true)
 |-- R_3: double (nullable = tru

In [11]:
df.show(5)

26/05/13 10:29:44 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+--------------------+----------+------------------+------------------+------------------+------------------+------------------+------------------+--------------------+------------------+----+----+-------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+----+------------------+------------------+------------------+------------------+------------------+--------------------+------------------+------------------+------------------+------------------+----+------------------+------------------+--------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+----+----+------------------+------------------

In [12]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [14]:
# Detectar columnas numericas
numeric_cols = [
    c for c, t in df.dtypes
    if t in ['double', 'int', 'bigint']
]

In [15]:
# Crear agregaciones
agg_exprs = []

for c in numeric_cols:
    agg_exprs.append(F.avg(c).alias(f"{c}_avg"))
    agg_exprs.append(F.max(c).alias(f"{c}_max"))
    agg_exprs.append(F.min(c).alias(f"{c}_min"))

In [16]:
# Agrupar por cliente
df_agg = df.groupBy("customer_ID").agg(*agg_exprs)

# Verificar tamano
df_agg.count()

458913

**union de labels **

In [17]:
# lectura de labels
labels = spark.read.csv(
    "hdfs://nn1:9000/amex/raw/train_labels.csv",
    header=True,
    inferSchema=True
)

In [18]:
# Join
data = df_agg.join(labels, on="customer_ID", how="inner")

In [19]:
# revisar dataset final
data.printSchema()
data.select("target").groupBy("target").count().show()

root
 |-- customer_ID: string (nullable = true)
 |-- P_2_avg: double (nullable = true)
 |-- P_2_max: double (nullable = true)
 |-- P_2_min: double (nullable = true)
 |-- D_39_avg: double (nullable = true)
 |-- D_39_max: double (nullable = true)
 |-- D_39_min: double (nullable = true)
 |-- B_1_avg: double (nullable = true)
 |-- B_1_max: double (nullable = true)
 |-- B_1_min: double (nullable = true)
 |-- B_2_avg: double (nullable = true)
 |-- B_2_max: double (nullable = true)
 |-- B_2_min: double (nullable = true)
 |-- R_1_avg: double (nullable = true)
 |-- R_1_max: double (nullable = true)
 |-- R_1_min: double (nullable = true)
 |-- S_3_avg: double (nullable = true)
 |-- S_3_max: double (nullable = true)
 |-- S_3_min: double (nullable = true)
 |-- D_41_avg: double (nullable = true)
 |-- D_41_max: double (nullable = true)
 |-- D_41_min: double (nullable = true)
 |-- B_3_avg: double (nullable = true)
 |-- B_3_max: double (nullable = true)
 |-- B_3_min: double (nullable = true)
 |-- D_42_

+------+------+
|target| count|
+------+------+
|     1|118828|
|     0|340085|
+------+------+



In [20]:
#guardar parquet
data.write.mode("overwrite").parquet(
    "hdfs://nn1:9000/amex/parquet/final_dataset"
)

26/05/13 11:37:34 WARN DAGScheduler: Broadcasting large task binary with size 1040.9 KiB


In [21]:
print(data.count())
print(len(data.columns))

data.groupBy("target").count().show()

ERROR:root:KeyboardInterrupt while sending command.                 (0 + 0) / 2]
Traceback (most recent call last):
  File "/home/mar-hadoop/amex_data/venv/lib/python3.10/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
  File "/home/mar-hadoop/amex_data/venv/lib/python3.10/site-packages/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
  File "/usr/lib/python3.10/socket.py", line 705, in readinto
    return self._sock.recv_into(b)
KeyboardInterrupt
26/05/13 11:59:59 WARN TaskSetManager: Lost task 20.0 in stage 30.0 (TID 688) (nn1 executor 1): java.net.ConnectException: Call From nn1/192.168.1.10 to nn1:9000 failed on connection exception: java.net.ConnectException: Conexión rehusada; For more details see:  http://wiki.apache.org/hadoop/ConnectionRefused
	at sun.reflect.NativeConstructorAccessorImpl.newInstance0(Native Method)
	at sun.reflect.NativeConstructorAccessorImp

KeyboardInterrupt: 

In [6]:
# recargar parquet
data = spark.read.parquet(
    "hdfs://nn1:9000/amex/parquet/final_dataset"
)

In [7]:
print(data.count())
print(len(data.columns))

data.groupBy("target").count().show()

458913
560


+------+------+
|target| count|
+------+------+
|     1|118828|
|     0|340085|
+------+------+



In [18]:
# 1. IMPORTACIONES
from pyspark.sql.functions import col, when, count, isnan
from pyspark.sql.types import DoubleType, IntegerType

from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import RandomForestClassifier

from pyspark.ml.evaluation import (
    MulticlassClassificationEvaluator,
    BinaryClassificationEvaluator
)

import time

In [19]:
# 2. IDENTIFICAR COLUMNAS NUMERICAS
numeric_cols = [
    f.name for f in data.schema.fields
    if isinstance(f.dataType, (DoubleType, IntegerType))
]


In [20]:
# 3. RELLENAR NULLS CON 0
# ==========================================
data = data.fillna(0, subset=numeric_cols)

In [21]:
# 4. VERIFICAR NULLS
# ==========================================
nulls = data.select([
    count(
        when(col(c).isNull() | isnan(c), c)
    ).alias(c)
    for c in numeric_cols
])

nulls.show()

26/05/14 07:57:50 WARN DAGScheduler: Broadcasting large task binary with size 1146.0 KiB


+-------+-------+-------+--------+--------+--------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+--------+--------+--------+-------+-------+-------+--------+--------+--------+--------+--------+--------+--------+--------+--------+-------+-------+-------+--------+--------+--------+-------+-------+-------+-------+-------+-------+--------+--------+--------+--------+--------+--------+--------+--------+--------+--------+--------+--------+-------+-------+-------+-------+-------+-------+-------+-------+-------+--------+--------+--------+--------+--------+--------+-------+-------+-------+-------+-------+-------+--------+--------+--------+-------+-------+-------+--------+--------+--------+--------+--------+--------+-------+-------+-------+--------+--------+--------+-------+-------+-------+--------+--------+--------+-------+-------+-------+-------+-------+-------+--------+--------+--------+-------+-------+-------+--------+--------+--------+------

In [22]:
# 5. TRAIN / TEST SPLIT
# ==========================================
train_data, test_data = data.randomSplit([0.8, 0.2], seed=42)

print("Train rows:", train_data.count())
print("Test rows :", test_data.count())


Train rows: 367429


Test rows : 91484


In [23]:
# 6. FEATURES
# ==========================================
feature_cols = [
    c for c in data.columns
    if c not in ["customer_ID", "target"]
]

print("Número de features:", len(feature_cols))


Número de features: 558


In [24]:
# 7. VECTOR ASSEMBLER
# ==========================================
assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features"
)

train_data = assembler.transform(train_data)
test_data = assembler.transform(test_data)


In [25]:
# 8. SELECCIONAR COLUMNAS NECESARIAS
# ==========================================
train_data = train_data.select("features", "target")
test_data = test_data.select("features", "target")


In [26]:
# 9. VERIFICAR QUE NO ESTE VACIO
# ==========================================
print("Train final:", train_data.count())
print("Test final :", test_data.count())

Train final: 367429


Test final : 91484


In [28]:
# 10. RANDOM FOREST
# ==========================================
rf = RandomForestClassifier(
    labelCol="target",
    featuresCol="features",
    numTrees=20,
    maxDepth=5,
    seed=42
)


In [29]:
# 11. ENTRENAMIENTO
# ==========================================
inicio = time.time()

model = rf.fit(train_data)

fin = time.time()

print("Tiempo entrenamiento (seg):", round(fin - inicio, 2))

Tiempo entrenamiento (seg): 1109.08


In [30]:
# 8. PREDICCIONES
predictions = model.transform(test_data)

predictions.select(
    "target",
    "prediction",
    "probability"
).show(10, truncate=False)

+------+----------+-----------------------------------------+
|target|prediction|probability                              |
+------+----------+-----------------------------------------+
|0     |0.0       |[0.9670520953078448,0.032947904692155236]|
|0     |0.0       |[0.9670520953078448,0.032947904692155236]|
|0     |0.0       |[0.8795606168132885,0.12043938318671146] |
|0     |0.0       |[0.68647652898077,0.31352347101923]      |
|0     |0.0       |[0.9670520953078448,0.032947904692155236]|
|0     |0.0       |[0.9670520953078448,0.032947904692155236]|
|0     |0.0       |[0.9670520953078448,0.032947904692155236]|
|0     |0.0       |[0.9456374917648089,0.05436250823519112] |
|1     |1.0       |[0.17957374031626985,0.8204262596837302] |
|0     |0.0       |[0.9566020948722024,0.04339790512779769] |
+------+----------+-----------------------------------------+
only showing top 10 rows



In [31]:
# 9. METRICAS

# Accuracy
accuracy_eval = MulticlassClassificationEvaluator(
    labelCol="target",
    predictionCol="prediction",
    metricName="accuracy"
)

accuracy = accuracy_eval.evaluate(predictions)


# Precision
precision_eval = MulticlassClassificationEvaluator(
    labelCol="target",
    predictionCol="prediction",
    metricName="weightedPrecision"
)

precision = precision_eval.evaluate(predictions)


# Recall
recall_eval = MulticlassClassificationEvaluator(
    labelCol="target",
    predictionCol="prediction",
    metricName="weightedRecall"
)

recall = recall_eval.evaluate(predictions)


# F1 Score
f1_eval = MulticlassClassificationEvaluator(
    labelCol="target",
    predictionCol="prediction",
    metricName="f1"
)

f1 = f1_eval.evaluate(predictions)


# AUC
auc_eval = BinaryClassificationEvaluator(
    labelCol="target",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

auc = auc_eval.evaluate(predictions)

In [32]:
# 10. RESULTADOS

print("\n===== RESULTADOS RANDOM FOREST =====")

print("Accuracy :", round(accuracy, 4))
print("Precision:", round(precision, 4))
print("Recall   :", round(recall, 4))
print("F1-Score :", round(f1, 4))
print("AUC ROC  :", round(auc, 4))


===== RESULTADOS RANDOM FOREST =====
Accuracy : 0.8668
Precision: 0.8635
Recall   : 0.8668
F1-Score : 0.8645
AUC ROC  : 0.9317
